# LSTM–GNN с фиксированным графом

Общая для всех вершин LSTM кодирует каждый ряд. Затем один графовый слой объединяет собственное состояние вершины и взвешенные состояния соседей. Граф оценивается по последним L наблюдениям train и замораживается.

Каждый запуск этого ноутбука обрабатывает **все три набора**: синтетику, MOEX и METR-LA.
Наборы загружаются по очереди. Вспомогательные функции находятся в `src/forecast_*.py`.
Результаты каждого опыта и сводный CSV сохраняются автоматически.

Нужны `numpy`, `pandas`, `torch`. Отдельные графовые библиотеки не требуются.

По умолчанию: один seed, CPU, до 15 эпох с ранней остановкой для каждого набора.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from forecast_batch import run_batch
from forecast_experiment import collect_results, fixed_graph
from forecast_models import build_adjacency

print("Проект:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)


Проект: /home/loktar/Desktop/money_coursework
PyTorch: 2.15.0+cpu


## Настройки серии

Каждый ноутбук запускает свою модель на `synthetic`, `moex`, `metr_la` по очереди.
История и горизонт читаются из `schema.json`; текущая постановка — 60 наблюдений и один шаг вперёд.
Используйте одинаковые `DATASETS`, `CONFIG`, `SEEDS`, `RUN_TAG` во всех четырёх ноутбуках.
Один seed — пилот; для оценки разброса позже задайте `[0, 1, 2]` во всех четырёх.

`EVALUATE_TEST=True`: после выбора лучшей эпохи по val рассчитывается test.
Test не участвует в обучении и выборе эпохи. Для новой настройки по val можно задать `False`.
Уже просмотренный test учитываем как исследовательскую оценку при дальнейшей настройке.


In [2]:
MODEL_KIND = "lstm_gnn_static"
DATASETS = ["synthetic", "moex", "metr_la"]
SEEDS = [0]
RUN_TAG = "pilot_v1"
EVALUATE_TEST = True

CONFIG = {
    "hidden_size": 16,
    "batch_size": 32,
    "max_epochs": 15,
    "patience": 4,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "gradient_clip": 1.0,
    "top_k": 3,
    "cpu_threads": 2,
    "device": "cpu",
    "min_delta": 0.0,
}

DATA_ROOT = PROJECT_ROOT / "data/processed"
RESULTS_ROOT = PROJECT_ROOT / "results"
print("Модель:", MODEL_KIND)
print("Наборы:", DATASETS)


Модель: lstm_gnn_static
Наборы: ['synthetic', 'moex', 'metr_la']


## Правило графа и архитектура

Рёбра выбираются top-k по $|\rho_{ij}|$ на входной истории; веса — те же абсолютные корреляции,
нормированные по строке. Диагональ нулевая. $A_{ij}$ означает сообщение от j к i.
Оба графовых варианта используют одинаковую длину истории оценки графа.


\[
h_i=\operatorname{LSTM}_{\theta}(x_{i,t-L+1:t}),\qquad
z_i=\operatorname{ReLU}\!\left(W_s h_i+W_n\sum_j A_{ij}h_j\right),\qquad
\widehat y_i=w^\top z_i+b.
\]

Собственный путь сохраняет информацию вершины даже при отсутствии рёбер.
Граф генератора синтетики моделям не передаётся: связи оцениваются по наблюдаемым значениям.
Динамический вариант меняет и рёбра, и веса; пока результат относится к обновлению графа в целом.
Режим с фиксированными рёбрами и обновляемыми весами предусмотрен в модуле для следующей проверки.

Ниже диагностика вызывается отдельно для каждого набора и использует только train и входные окна val.


## Общий загрузчик и покрытие целей

CSV каждого набора читаются один раз при его запуске. Масштабирование каждого ряда обучается только по наблюдаемым train-значениям.
Пропуски входа заполняются прошлым значением только внутри выбранного окна, затем обучающим средним.
Цели остаются пропущенными; `y_mask` применяется и в функции потерь, и во всех метриках.

Таблица ниже показывает **фактическое число оцениваемых целей** каждой вершины.
Вершина с нулевым покрытием остаётся в составе, но не получает выдуманную нулевую ошибку.
Состав акций по доступности test не пересматривается.

Функция ниже вызывается автоматически внутри цикла по наборам.


In [3]:
def inspect_dataset(data, dataset_name):
    # Keep inspection local so the previous dataset can be released.
    display(pd.DataFrame([
        {"dataset": dataset_name, "split": split, "samples": len(data.dataset(split)),
         "nodes": len(data.nodes), "history": data.history, "features_per_node": 1}
        for split in ("train", "val", "test")
    ]))
    coverage = data.quality().pivot(index="node", columns="split", values="evaluated_targets")
    display(coverage.head(12))
    zero_coverage = coverage.loc[coverage.eq(0).any(axis=1)]
    if len(zero_coverage):
        print("Вершины без оцениваемых целей хотя бы в одном разбиении:")
        display(zero_coverage)
    example = data.dataset("val")[0]
    print("Один вход:", example["X"].shape, "цель:", example["y_norm"].shape)

    static_A, graph_info = fixed_graph(data, CONFIG["top_k"])
    print("Фиксированный граф:", graph_info)
    display(pd.DataFrame(static_A.numpy(), index=data.nodes, columns=data.nodes).iloc[:6, :6])
    diagnostics = []
    validation = data.dataset("val")
    for index in sorted({0, len(validation) // 2, len(validation) - 1}):
        sample = validation[index]
        dynamic_A = build_adjacency(torch.from_numpy(sample["X"]), CONFIG["top_k"])
        fixed_edges, current_edges = static_A > 0, dynamic_A > 0
        union = int((fixed_edges | current_edges).sum())
        intersection = int((fixed_edges & current_edges).sum())
        diagnostics.append({
            "origin_time": str(data.times[int(sample["origin_row"])]),
            "edge_jaccard_vs_static": intersection / union if union else 1.0,
            "mean_absolute_weight_change": float((static_A - dynamic_A).abs().mean()),
        })
    display(pd.DataFrame(diagnostics))


## Запуск на трёх наборах

Нейросети обучаются по MSE нормированных целей с маской. Лучший checkpoint выбирается только по val:
по MAE для синтетики и METR-LA, по QLIKE для MOEX. У всех нейросетей одинаковые правила оптимизации,
ранней остановки и перемешивания обучающих примеров.

MAE/RMSE считаются в исходной шкале. Для MOEX прогноз log(RV) переводится через exp в RV;
QLIKE используется в форме `log(v_hat) + RV/v_hat`. Её значение может быть отрицательным;
сравнивать его можно на одинаковых наблюдениях.

Повторный запуск той же конфигурации переиспользует проверенный результат.
Для повторения с другой инициализацией меняйте seed. `RUN_TAG` помогает разделять серии опытов.

После каждого завершённого опыта обновляется `summary.csv` этой серии.
В `status.csv` виден результат каждого набора и seed; `errors.csv` содержит ошибки.
При ошибке следующий набор всё равно запускается; в конце частичная серия явно отмечается.
Повторный запуск переиспользует **завершённые** опыты с той же конфигурацией.
Прерванное обучение текущий runner не возобновляет: для него потребуется новый `RUN_TAG`.


In [4]:
batch = run_batch(
    DATA_ROOT, MODEL_KIND, RESULTS_ROOT,
    datasets=DATASETS, seeds=SEEDS, config=CONFIG,
    evaluate_test=EVALUATE_TEST, run_tag=RUN_TAG,
    inspect_dataset=inspect_dataset,
)
reports = batch["reports"]
summary = batch["summary"]
SUMMARY_DIR = batch["summary_dir"]
display(batch["status"])
display(summary)
print("Сводные результаты:", SUMMARY_DIR / "summary.csv")



=== lstm_gnn_static: synthetic ===


,dataset,split,samples,nodes,history,features_per_node
0,synthetic,train,4140,12,60,1
1,synthetic,val,600,12,60,1
2,synthetic,test,1201,12,60,1


split,test,train,val
node,,,
node_0,1201,4140,600
node_1,1201,4140,600
node_10,1201,4140,600
node_11,1201,4140,600
node_2,1201,4140,600
node_3,1201,4140,600
node_4,1201,4140,600
node_5,1201,4140,600
node_6,1201,4140,600


Один вход: (60, 12, 1) цель: (12,)
Фиксированный граф: {'anchor_origin_row': 4199, 'anchor_time': '4199', 'history': 60, 'rule': 'row-normalized top-k absolute Pearson correlation', 'A_ij': 'message from node j to node i', 'self_edges': False, 'estimated_on': 'last history-length window within train only'}


,node_0,node_1,node_2,node_3,node_4,node_5
node_0,0.000000,0.000000,0.420999,0.000000,0.0,0.000000
node_1,0.000000,0.000000,0.000000,0.313441,0.0,0.345814
node_2,0.427015,0.000000,0.000000,0.000000,0.0,0.000000
node_3,0.000000,0.364898,0.000000,0.000000,0.0,0.405260
node_4,0.265694,0.000000,0.000000,0.426528,0.0,0.307778
node_5,0.000000,0.000000,0.000000,0.000000,0.0,0.000000


,origin_time,edge_jaccard_vs_static,mean_absolute_weight_change
0,4199,1.000000,0.000000
1,4499,0.142857,0.127282
2,4798,0.180328,0.119343


... (14 outputs left out)

lstm_gnn_static seed=0 epoch=1: train MSE=0.32479; val mae=2.52185
lstm_gnn_static seed=0 epoch=2: train MSE=0.24511; val mae=2.42499
lstm_gnn_static seed=0 epoch=3: train MSE=0.24181; val mae=2.38756
lstm_gnn_static seed=0 epoch=4: train MSE=0.23886; val mae=2.37675
lstm_gnn_static seed=0 epoch=5: train MSE=0.23616; val mae=2.39874
lstm_gnn_static seed=0 epoch=6: train MSE=0.23324; val mae=2.39483
lstm_gnn_static seed=0 epoch=7: train MSE=0.23046; val mae=2.37708
lstm_gnn_static seed=0 epoch=8: train MSE=0.22875; val mae=2.32493
lstm_gnn_static seed=0 epoch=9: train MSE=0.22775; val mae=2.324
lstm_gnn_static seed=0 epoch=10: train MSE=0.22695; val mae=2.29914
lstm_gnn_static seed=0 epoch=11: train MSE=0.22651; val mae=2.36891
lstm_gnn_static seed=0 epoch=12: train MSE=0.22605; val mae=2.30652
lstm_gnn_static seed=0 epoch=13: train MSE=0.22579; val mae=2.33786
lstm_gnn_static seed=0 epoch=14: train MSE=0.22548; val mae=2.34639
Saved: /home/loktar/Desktop/money_coursework/results/metr_l

,dataset,model,seed,status,error,output_dir
0,synthetic,lstm_gnn_static,0,completed,,/home/loktar/Desktop/money_coursework/results/...
1,moex,lstm_gnn_static,0,completed,,/home/loktar/Desktop/money_coursework/results/...
2,metr_la,lstm_gnn_static,0,completed,,/home/loktar/Desktop/money_coursework/results/...


,dataset,model,seed,split,primary_metric,primary_score,mae,rmse,qlike,normalized_mse,n_observed,n_origins,best_epoch,parameters,fit_seconds,comparison_id,run_id,output_dir
0,synthetic,lstm_gnn_static,0,val,mae,0.081928,0.081928,0.102395,NaN,0.809143,7200,600,4,1761,17.240769,36989cce73a0,ad06a7c012ff,/home/loktar/Desktop/money_coursework/results/...
1,synthetic,lstm_gnn_static,0,test,mae,0.081709,0.081709,0.102267,NaN,0.807098,14412,1201,4,1761,17.240769,36989cce73a0,ad06a7c012ff,/home/loktar/Desktop/money_coursework/results/...
2,moex,lstm_gnn_static,0,val,qlike,-7.465629,0.000134,0.000290,-7.465629,0.521630,2371,252,14,1761,6.221492,362a12d77352,946639243f4f,/home/loktar/Desktop/money_coursework/results/...
3,moex,lstm_gnn_static,0,test,qlike,-7.280866,0.000146,0.000583,-7.280866,0.481681,2283,254,14,1761,6.221492,362a12d77352,946639243f4f,/home/loktar/Desktop/money_coursework/results/...
4,metr_la,lstm_gnn_static,0,val,mae,2.299141,2.299141,3.943389,NaN,0.225813,660000,3246,10,1761,2616.135389,c43da30bee45,708560d0560e,/home/loktar/Desktop/money_coursework/results/...
5,metr_la,lstm_gnn_static,0,test,mae,2.483712,2.483712,4.205260,NaN,0.244837,1247252,6189,10,1761,2616.135389,c43da30bee45,708560d0560e,/home/loktar/Desktop/money_coursework/results/...


Сводные результаты: /home/loktar/Desktop/money_coursework/results/summaries/lstm_gnn_static/20261124T081937_356430Z/summary.csv


## Сохранённые результаты и примеры

Подробные результаты: `results/<dataset>/<model>/seed_<seed>/<run_id>/`.
Там находятся `run.json`, метрики, ошибки по датам `*_losses.csv`, метрики по вершинам,
прогнозы с исходной маской в `*_predictions.npz`. У нейросетей также сохраняются история обучения и лучшая модель.

Сводка этой серии: `results/summaries/<model>/<дата_UTC>/summary.csv`.
Одна строка — один набор, seed и разбиение val/test; при одном seed и трёх успешных наборах получится 6 строк.
В сводке есть основная метрика, MAE, RMSE, QLIKE для MOEX, число целей, время обучения и путь к опыту.
Разные серии получают отдельные папки. Ниже показаны примеры для каждого успешно обработанного набора.


In [5]:
preview_split = "test" if EVALUATE_TEST else "val"
for dataset_name in DATASETS:
    dataset_reports = [report for report in reports if report["dataset"] == dataset_name]
    if not dataset_reports:
        continue
    run_dir = Path(dataset_reports[0]["output_dir"])
    print(dataset_name, "—", run_dir)
    display(pd.read_csv(run_dir / f"{preview_split}_per_node.csv").head(12))
    with np.load(run_dir / f"{preview_split}_predictions.npz") as saved:
        preview = pd.DataFrame({
            "node": saved["node_order"], "observed": saved["mask"][0],
            "actual_raw": saved["target_raw"][0], "prediction_raw": saved["prediction_raw"][0],
        })
    display(preview.head(12))


synthetic — /home/loktar/Desktop/money_coursework/results/synthetic/lstm_gnn_static/seed_000/ad06a7c012ff


,node_id,node,name,observed_pairs,n_observed,mae,rmse
0,0,node_0,node_0,1201,1201,0.081801,0.103363
1,1,node_1,node_1,1201,1201,0.078106,0.097476
2,2,node_2,node_2,1201,1201,0.082110,0.103237
3,3,node_3,node_3,1201,1201,0.082873,0.101850
4,4,node_4,node_4,1201,1201,0.080942,0.101906
5,5,node_5,node_5,1201,1201,0.080648,0.101189
6,6,node_6,node_6,1201,1201,0.082647,0.104610
7,7,node_7,node_7,1201,1201,0.081871,0.101517
8,8,node_8,node_8,1201,1201,0.081086,0.102336
9,9,node_9,node_9,1201,1201,0.084307,0.103975


,node,observed,actual_raw,prediction_raw
0,node_0,True,-0.018677,-0.014774
1,node_1,True,0.002347,0.004353
2,node_2,True,-0.135561,0.008895
3,node_3,True,0.063401,0.038326
4,node_4,True,0.048733,-0.011517
5,node_5,True,-0.123078,-0.029866
6,node_6,True,-0.013832,-0.055677
7,node_7,True,-0.040185,-0.006758
8,node_8,True,0.174519,0.054455
9,node_9,True,-0.034389,-0.018240


moex — /home/loktar/Desktop/money_coursework/results/moex/lstm_gnn_static/seed_000/946639243f4f


,node_id,node,name,observed_pairs,n_observed,mae,rmse,qlike
0,0,SBER,SBER,254,254,0.000076,0.000148,-8.049046
1,1,GAZP,GAZP,254,254,0.000168,0.000321,-7.102832
2,2,LKOH,LKOH,254,254,0.000099,0.000204,-7.620436
3,3,GMKN,GMKN,254,254,0.000125,0.000192,-7.130637
4,4,YNDX,YNDX,0,0,NaN,NaN,NaN
5,5,ROSN,ROSN,254,254,0.000113,0.000189,-7.405433
6,6,PLZL,PLZL,251,251,0.000140,0.000266,-7.185873
7,7,MGNT,MGNT,254,254,0.000138,0.000248,-7.304306
8,8,NVTK,NVTK,254,254,0.000191,0.000349,-6.855679
9,9,VTBR,VTBR,254,254,0.000260,0.001600,-6.872427


,node,observed,actual_raw,prediction_raw
0,SBER,True,0.000196,0.000146
1,GAZP,True,0.000641,0.000260
2,LKOH,True,0.000351,0.000145
3,GMKN,True,0.000510,0.000290
4,YNDX,False,NaN,0.000350
5,ROSN,True,0.000478,0.000200
6,PLZL,True,0.000058,0.000117
7,MGNT,True,0.000848,0.000437
8,NVTK,True,0.000861,0.000315
9,VTBR,True,0.000401,0.000235


metr_la — /home/loktar/Desktop/money_coursework/results/metr_la/lstm_gnn_static/seed_000/708560d0560e


,node_id,node,name,observed_pairs,n_observed,mae,rmse
0,0,773869,773869,6188,6188,1.942467,3.673721
1,1,767541,767541,6189,6189,1.547502,2.447426
2,2,767542,767542,6189,6189,1.800689,3.216200
3,3,717447,717447,6185,6185,2.494810,3.892245
4,4,717446,717446,5963,5963,3.616442,5.123271
5,5,717445,717445,6185,6185,3.457153,5.239358
6,6,773062,773062,6185,6185,2.477260,3.914498
7,7,767620,767620,6186,6186,1.665699,2.440862
8,8,737529,737529,6088,6088,2.377029,3.887306
9,9,717816,717816,6186,6186,2.539112,4.501756


,node,observed,actual_raw,prediction_raw
0,773869,True,64.750,64.528969
1,767541,True,56.375,60.039632
2,767542,True,69.375,67.430406
3,717447,True,58.000,58.204597
4,717446,True,66.000,65.218009
5,717445,True,44.375,53.654729
6,773062,True,63.125,64.839717
7,767620,True,52.250,61.088419
8,737529,True,65.750,61.750501
9,717816,True,68.000,67.055892


## Сопоставление четырёх запусков

Сравнивайте строки с одним `comparison_id`: он фиксирует данные, нормировку, общие настройки и версии кода.
Метрики разных наборов данных имеют разные единицы и напрямую не сопоставляются.

Обычная LSTM здесь многомерная, а LSTM–GNN использует общий по вершинам временной кодировщик.
Поэтому их сравнение относится к двум семействам моделей, а не к изолированному добавлению одного слоя.
Главная проверка обновления графа — **static против dynamic** при одинаковых seed и настройках.

Текущая синтетика со статическим истинным графом служит контрольным сценарием.
Для проверки пользы изменения топологии позже потребуется синтетика с меняющимися связями.

`comparison.csv` собирает уже завершённые запуски других моделей с теми же протоколом и seed
внутри каждого набора. После запуска всех четырёх ноутбуков таблица последнего включает все четыре модели.
При одном seed и наличии val/test это 24 строки. Ошибки разных наборов не усредняются.


In [6]:
comparison_parts = []
for dataset_name in DATASETS:
    current_ids = summary.loc[summary["dataset"].eq(dataset_name), "comparison_id"].unique()
    for split in (("val", "test") if EVALUATE_TEST else ("val",)):
        table = collect_results(RESULTS_ROOT, dataset_name, split=split, run_tag=RUN_TAG)
        if table.empty:
            continue
        table = table.loc[table["comparison_id"].isin(current_ids) & table["seed"].isin(SEEDS)].copy()
        if not table.empty:
            table.insert(0, "dataset", dataset_name)
            comparison_parts.append(table)
comparison = pd.concat(comparison_parts, ignore_index=True) if comparison_parts else pd.DataFrame(
    columns=["dataset", "model", "seed", "split", "comparison_id", "mae", "rmse", "qlike"]
)
comparison.to_csv(SUMMARY_DIR / "comparison.csv", index=False)
display(comparison)
print("Сопоставимые запуски:", SUMMARY_DIR / "comparison.csv")

failed = batch["status"].loc[batch["status"]["status"].ne("completed")]
if not failed.empty:
    display(failed)
    raise RuntimeError(f"Серия завершена частично. Ошибки: {SUMMARY_DIR / 'errors.csv'}")
print("Готово: результаты сохранены для всех трёх наборов.")


,dataset,model,seed,split,comparison_id,mae,rmse,n_observed,best_epoch,parameters,fit_seconds,output_dir,qlike
0,synthetic,lstm,0,val,36989cce73a0,0.081100,0.101304,7200,14.0,2124,11.641013,/home/loktar/Desktop/money_coursework/results/...,NaN
1,synthetic,lstm_gnn_static,0,val,36989cce73a0,0.081928,0.102395,7200,4.0,1761,17.240769,/home/loktar/Desktop/money_coursework/results/...,NaN
2,synthetic,persistence,0,val,36989cce73a0,0.096973,0.121423,7200,NaN,0,0.000000,/home/loktar/Desktop/money_coursework/results/...,NaN
3,synthetic,lstm,0,test,36989cce73a0,0.080639,0.101053,14412,14.0,2124,11.641013,/home/loktar/Desktop/money_coursework/results/...,NaN
4,synthetic,lstm_gnn_static,0,test,36989cce73a0,0.081709,0.102267,14412,4.0,1761,17.240769,/home/loktar/Desktop/money_coursework/results/...,NaN
5,synthetic,persistence,0,test,36989cce73a0,0.096982,0.121153,14412,NaN,0,0.000000,/home/loktar/Desktop/money_coursework/results/...,NaN
6,moex,lstm,0,val,362a12d77352,0.000145,0.000304,2371,15.0,1962,2.750675,/home/loktar/Desktop/money_coursework/results/...,-7.398339
7,moex,lstm_gnn_static,0,val,362a12d77352,0.000134,0.000290,2371,14.0,1761,6.221492,/home/loktar/Desktop/money_coursework/results/...,-7.465629
8,moex,persistence,0,val,362a12d77352,0.000167,0.000364,2371,NaN,0,0.000000,/home/loktar/Desktop/money_coursework/results/...,-7.368911
9,moex,lstm,0,test,362a12d77352,0.000155,0.000589,2283,15.0,1962,2.750675,/home/loktar/Desktop/money_coursework/results/...,-7.227486


Сопоставимые запуски: /home/loktar/Desktop/money_coursework/results/summaries/lstm_gnn_static/20261124T081937_356430Z/comparison.csv
Готово: результаты сохранены для всех трёх наборов.
